# Chemotion API Client & ELN Export Demo

This notebook demonstrates using Chemotion API Python client's core functionality, including Pydantic v2 validation, stoichiometry mapping, and RO-Crate 1.1 `.eln` archive exporting.

### 1. Connection and Authentication

In [ ]:
import sys
sys.path.insert(0, ".")

from chemotion_api import Instance

# Establish connection using simplified inline credentials (ideal for notebook sessions)
# Note: For automated scripts, you can also use: with Instance(...) as instance:
instance = Instance("https://my.chemotion.instance/", username="MM", password="PASSWORD")
print("Connected & Authenticated to Chemotion ELN.")

Connected & Authenticated to Chemotion ELN.


### 2. Create and Save a Sample (Pydantic v2 Validation)

In [10]:
# Fetch or create a project collection directly from the Instance
project_col = instance.get_or_create_collection("Notebook Demo")

# Create a sample using SMILES
sample = project_col.new_sample_smiles("CCO")
sample.name = "Ethanol sample"
sample.properties.description = "Ethanol for esterification reaction"  # Attribute-style access
sample.properties.purity = 0.99  # Validated by Pydantic
sample.save()
print(f"Sample successfully created with ID: {sample.id}")

Sample successfully created with ID: 1667


### 3. Create a Reaction (Yield Caching & Product Equivalent Mapping)

In [11]:
# Create a new reaction in the collection
reaction = project_col.new_reaction()
reaction.name = "Demo reaction"
reaction.properties.description = "Synthesizing water"

# Set yield percentage
reaction.properties.yield_value = 92.5
reaction.save()
print(f"Reaction saved with cached yield: {reaction.properties.yield_value}% (maps to product equivalent)")

Reaction saved with cached yield: 92.5% (maps to product equivalent)


### 4. Attach Analytical Datasets

In [12]:
# Add analysis NMR dataset
test_file = "tests/data/SG-V3545-6-13.cif"
sample.analyses.add_analyses("NMR").add_dataset(test_file)
sample.save()
print(f"Analysis dataset '{test_file}' successfully attached to sample.")

Analysis dataset 'tests/data/SG-V3545-6-13.cif' successfully attached to sample.


### 5. Add Authors to the RO-Crate Metadata

In [13]:
from chemotion_api.authors import Author, AuthorAffiliation

project_col.set_authors([
    Author(name="Ada Lovelace", orcid="0000-0002-1825-0097",
            affiliations=[AuthorAffiliation(name="Example University",
                                            ror="https://ror.org/05dxps055")]),
])

### 6. Export Collection to RO-Crate (.eln ZIP Archive)

In [14]:
import os
export_path = "notebook_export.eln"

# Run collection exporter
project_col.export_eln(export_path)
print(f"Collection successfully exported to: {os.path.abspath(export_path)}")

Collection successfully exported to: /home/harivyasi/work/projects/API2/chemotionapi/notebook_export.eln


### 6. Inspect the Exported RO-Crate Contents

In [15]:
import zipfile
import json

with zipfile.ZipFile(export_path, 'r') as z:
    print("ZIP namelist:")
    for name in z.namelist():
        print(f" - {name}")
        
    # Load metadata file
    meta_entry = "notebook_export.eln/ro-crate-metadata.json"
    if meta_entry in z.namelist():
        metadata_content = z.read(meta_entry).decode('utf-8')
        metadata = json.loads(metadata_content)
        print("\nMetadata context:")
        print(metadata.get("@context"))
        print("\nMetadata graph sample nodes:")
        for node in metadata.get("@graph", []):
            if "@type" in node:
                print(f" - ID: {node.get('@id')} | Type: {node.get('@type')} | Name: {node.get('name')}")

ZIP namelist:
 - notebook_export.eln/ro-crate-metadata.json

Metadata context:
https://w3id.org/ro/crate/1.1/context

Metadata graph sample nodes:
 - ID: ./samples/MM-2_1667/ | Type: ['Dataset', 'http://semanticscience.org/resource/SIO_001378', 'http://purl.obolibrary.org/obo/CHEBI_59999'] | Name: None
 - ID: _:N0f9e728fa5234978a396abf5c0abf111 | Type: http://www.w3.org/ns/adms#Identifier | Name: None
 - ID: _:Nb0e21624c2974d2bb77e88b5fc615097 | Type: http://purl.obolibrary.org/obo/CHMO_0002820 | Name: None
 - ID: #molecule_1120 | Type: http://purl.obolibrary.org/obo/CHEBI_23367 | Name: None
 - ID: _:Ndee40703d3e94bf7aade5fe0116b0c3d | Type: http://semanticscience.org/resource/CHEMINF_000018 | Name: None
 - ID: _:N27a7dd0ec9cc4b0ebbf359604c8d560e | Type: http://semanticscience.org/resource/CHEMINF_000018 | Name: None
 - ID: _:N39e8a500161a49739f897bef2033eb52 | Type: http://semanticscience.org/resource/CHEMINF_000059 | Name: None
 - ID: _:Ncb72f4d9d8cc463796c29ba0dc134272 | Type: http: